In [1]:
import pyodbc
import pandas as pd
import numpy as np
from unidecode import unidecode

In [2]:

pip install Unidecode

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: E:\E2E-ECommerce-Analytics\venv\Scripts\python.exe -m pip install --upgrade pip


In [3]:
# ESTABLEISH THE CONNECTION

server = r'DESKTOP-QKR9KDS\SQLEXPRESS'
database  = "olistEcommProject"
driver = '{ODBC Driver 17 for SQL Server}'

connection_string = (
    f'DRIVER={driver};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;' # This specifies Windows Authentication
)

conn = pyodbc.connect(connection_string)

In [4]:
query = "select * from olist_customers_dataset"
customers = pd.read_sql(query, conn)


C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\1956739419.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(query, conn)


### olist_customers_dataset

In [5]:
customers.sample(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
91945,ec3ec417b16fb18d288705ec41bdd297,288ae9eef8aea0a10244894a9688b226,13327,salto,SP
17733,2dbef7c4fd8693011193adc8ae646272,9c8c4083662be572f6bb390ae17ebd9e,5011,sao paulo,SP
99321,ffb0461e1e70c023890d66303b1d3d4b,6c24363a631b5b08a9ece392fa118f97,71225,brasilia,DF
3406,08e5ed5ac7e17524347b55e9db5e8692,6e0f804fbab9756e54970777c3de50f2,13050,campinas,SP
57857,952f3c5b88acea4882984111a06c8dfa,374375f8a47d9782455287c7b799ed26,88470,rancho queimado,SC
8292,157d0e216493fbaacd5ea9175f15076b,171485408014aed36c75286eb7bb0535,54786,camaragibe,PE
90719,e904e6fe5895cf401fd990b33c24ccb4,7589544c08cc5de364cad85ee73f4b63,2636,sao paulo,SP
32349,53849dfbee2387632eef6ed7f4deb3dc,273a1f5b9953aac34f2c846403614771,92420,canoas,RS
51330,8450bc4f0cf0fba040163c4024ae86d1,2a9a9283ac70bd5f1a56e6747ddbd906,17120,agudos,SP
60430,9ba493187a89070e398b4fe8bc663f8a,1ae27253a2f4e1a9584803148d78da8c,38702,patos de minas,MG


In [6]:
customers['customer_city'] = customers['customer_city'].str.title()

In [7]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [8]:
customers['customer_id'].nunique()

99441

In [9]:
customers['customer_unique_id'].nunique()

96096

In [10]:
sum(customers['customer_unique_id']== customers['customer_id']) # this means no id is matching with eacher other we can drop one of the column.

0

### olist_geolocation_dataset

In [11]:
query = "select * from olist_geolocation_dataset"
geo = pd.read_sql(query, conn)


C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\340750199.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  geo = pd.read_sql(query, conn)


In [12]:
geo.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1,01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
2,01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
3,01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
4,01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


In [13]:
geo['geolocation_city'] = geo['geolocation_city'].str.title()

In [14]:
geo.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.54562128115268,-46.63929204800168,Sao Paulo,SP
1,01046,-23.546081127035535,-46.64482029837157,Sao Paulo,SP
2,01046,-23.54612896641469,-46.64295148361138,Sao Paulo,SP
3,01041,-23.5443921648681,-46.63949930627844,Sao Paulo,SP
4,01035,-23.541577961711493,-46.64160722329613,Sao Paulo,SP


In [15]:
geo.duplicated().sum()

np.int64(261831)

In [16]:
geo.drop_duplicates(inplace = True)

In [17]:
geo.shape

(738332, 5)

In [18]:
geo.drop(columns = ['geolocation_lat','geolocation_lng'], inplace = True)

In [19]:
geo['geolocation_city'] = geo['geolocation_city'].apply(lambda x: unidecode(x))

In [20]:
geo['geolocation_city'] = geo['geolocation_city'].str.title()

In [21]:
def contain_numbers(s):
    return any( i.isdigit() for i in str(s))

In [22]:
geo[geo['geolocation_city'].apply(contain_numbers)]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
200270,29735,Quilometro 14 Do Mutum,ES
377770,17970,Sao Joao Do Pau D%26Apos%3Balho,SP
539465,29735,Quilometro 14 Do Mutum,ES
726758,57010,Maceia3,AL
786179,71880,Riacho Fundo 2,DF
823386,78278,Lambari D%26Apos%3Boeste,MT
896733,87365,4O. Centenario,PR
896977,87365,4O Centenario,PR
897217,87365,4O Centenario,PR
977775,96130,Colonia Z-3,RS


In [23]:
geo['geolocation_city'] = geo['geolocation_city'].str.replace(r'\d+', '', regex=True)

In [24]:
geo[geo['geolocation_city'].apply(contain_numbers)]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state


In [25]:
geo[geo['geolocation_zip_code_prefix'] == '17970']

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
377365,17970,Sao Joao Do Pau D'Alho,SP
377387,17970,Sao Joao Do Pau D'Alho,SP
377465,17970,Sao Joao Do Pau D'Alho,SP
377497,17970,Sao Joao Do Pau D Alho,SP
377582,17970,Sao Joao Do Pau D Alho,SP
377633,17970,Sao Joao Do Pau D'Alho,SP
377642,17970,Sao Joao Do Pau D'Alho,SP
377652,17970,Sao Joao Do Pau Dalho,SP
377698,17970,Sao Joao Do Pau D'Alho,SP
377746,17970,Sao Joao Do Pau Dalho,SP


In [26]:
geo['geolocation_city'] = geo['geolocation_city'].str.replace("'", '', regex=True) # remove the inverted commas

In [27]:
geo[geo['geolocation_zip_code_prefix'] == '17970']

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
377365,17970,Sao Joao Do Pau DAlho,SP
377387,17970,Sao Joao Do Pau DAlho,SP
377465,17970,Sao Joao Do Pau DAlho,SP
377497,17970,Sao Joao Do Pau D Alho,SP
377582,17970,Sao Joao Do Pau D Alho,SP
377633,17970,Sao Joao Do Pau DAlho,SP
377642,17970,Sao Joao Do Pau DAlho,SP
377652,17970,Sao Joao Do Pau Dalho,SP
377698,17970,Sao Joao Do Pau DAlho,SP
377746,17970,Sao Joao Do Pau Dalho,SP


In [28]:
geo['geolocation_city'] = geo['geolocation_city'].str.replace("*", '', regex=False)

In [29]:
geo[geo['geolocation_zip_code_prefix'].str.startswith('*')]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state


In [30]:
import re

In [31]:
def keep_only_text(x):
    if pd.isna(x):
        return x
    return re.sub(r"[^A-Za-z\s]", "", str(x)).strip()
    

In [32]:
geo['geolocation_city'] = geo['geolocation_city'].apply(keep_only_text)

In [33]:
geo.drop_duplicates(inplace = True)

In [34]:
geo

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
0,01037,Sao Paulo,SP
1,01046,Sao Paulo,SP
3,01041,Sao Paulo,SP
4,01035,Sao Paulo,SP
5,01012,Sao Paulo,SP
...,...,...,...
999774,99955,Vila Langaro,RS
999780,99970,Ciriaco,RS
999786,99910,Floriano Peixoto,RS
999803,99920,Erebango,RS


In [35]:
customers['customer_city'] = customers['customer_city'].apply(lambda x: unidecode(x))

In [36]:
customers['customer_city'] = customers['customer_city'].apply(keep_only_text)

In [37]:
customers.duplicated().sum()

np.int64(0)

### olist_sellers_dataset

In [38]:
query = "select * from olist_sellers_dataset"
seller = pd.read_sql(query, conn)


C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\589116168.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seller = pd.read_sql(query, conn)


In [39]:
seller['seller_city'] = seller['seller_city'].str.title()

In [40]:
seller.head(5)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,Sao Paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP


In [41]:
seller['seller_city'] = seller['seller_city'].apply(keep_only_text)

In [42]:
seller.head(5)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,Sao Paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP


### olist_orders_dataset

In [112]:
query = "select * from olist_orders_dataset"
orders = pd.read_sql(query, conn)


C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\52118180.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql(query, conn)


In [113]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [45]:
orders = orders[
    ((orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds()/86400 >= 0) &
    ((orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.days >= 0) 
]

In [46]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


In [47]:
to_remove = orders[
    (orders['order_status'] == 'canceled') &
    (orders['order_delivered_customer_date'].notna())
].index

In [48]:
orders = orders.drop(to_remove)

In [49]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


### olist_orders_dataset

In [50]:
query = "select * from olist_order_items_dataset"
olist_items = pd.read_sql(query, conn)

C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\1154614543.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  olist_items = pd.read_sql(query, conn)


In [51]:
olist_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.900002,13.290000
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.899994,19.930000
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.000000,17.870001
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.990000,12.790000
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.899994,18.139999
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.989990,43.410000
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.000000,36.529999
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.900002,16.950001
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.990002,8.720000


In [52]:
print(orders['order_id'].nunique())
print(olist_items['order_id'].nunique())

96279
98666


In [53]:
orders = pd.merge(orders, olist_items, on='order_id', how = 'inner')

In [54]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109883 entries, 0 to 109882
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       109883 non-null  object        
 1   customer_id                    109883 non-null  object        
 2   order_status                   109883 non-null  object        
 3   order_purchase_timestamp       109883 non-null  datetime64[ns]
 4   order_approved_at              109883 non-null  datetime64[ns]
 5   order_delivered_carrier_date   109883 non-null  datetime64[ns]
 6   order_delivered_customer_date  108631 non-null  datetime64[ns]
 7   order_estimated_delivery_date  109883 non-null  datetime64[ns]
 8   order_item_id                  109883 non-null  int64         
 9   product_id                     109883 non-null  object        
 10  seller_id                      109883 non-null  object        
 11  

In [55]:
orders = orders[(orders['shipping_limit_date'] > orders['order_purchase_timestamp']) & 
(orders['shipping_limit_date'] > orders['order_approved_at']) &
(orders['shipping_limit_date'] < orders['order_delivered_customer_date'])]

In [56]:
orders.sort_values(by =['order_item_id', 'order_id'], ascending = False)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
98094,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,21,79ce45dbc2ea29b22b5a261bbb7b7ee7,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,7.800000,6.570000
103896,ab14fdcfbe524636d65ee38360e22ce8,bd5d39761aa56689a265d95d8d32b8be,delivered,2017-08-23 09:22:34,2017-08-24 14:30:23,2017-08-25 20:07:36,2017-09-02 12:13:03,2017-09-21,20,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-08-30 14:30:23,98.699997,14.440000
98093,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,20,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.200000,7.890000
96011,1b15974a0141d54e36626dca3fdc731a,be1b70680b9f9694d8c70f41fa3dc92b,delivered,2018-02-22 15:30:41,2018-02-24 03:20:27,2018-03-02 00:18:01,2018-03-05 15:22:27,2018-03-08,20,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,2018-03-01 02:50:48,100.000000,10.120000
103895,ab14fdcfbe524636d65ee38360e22ce8,bd5d39761aa56689a265d95d8d32b8be,delivered,2017-08-23 09:22:34,2017-08-24 14:30:23,2017-08-25 20:07:36,2017-09-02 12:13:03,2017-09-21,19,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-08-30 14:30:23,98.699997,14.440000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108902,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,delivered,2017-12-10 11:53:48,2017-12-10 12:10:31,2017-12-12 01:07:48,2017-12-18 22:03:38,2018-01-04,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.900000,11.850000
5741,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.899994,18.139999
6905,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.000000,17.870001
79365,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.899994,19.930000


In [111]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'price', 'product_id', 'seller_id', 'shipping_limit_date', 'new_price',
       'freight_value', 'quantity'],
      dtype='object')

In [57]:
orders[orders['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
98074,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98075,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98076,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98077,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98078,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98079,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98080,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98081,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98082,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
98083,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [58]:
orders.drop(columns = 'order_item_id', inplace= True)

In [114]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [108]:
orders = orders.groupby(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
        'price', 'product_id', 'seller_id',
       'shipping_limit_date']).agg(
    new_price = ('price', 'sum'),
    freight_value = ('freight_value', 'sum'),
    quantity = ('product_id', 'count')
       ).reset_index()

In [115]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


In [110]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'price', 'product_id', 'seller_id', 'shipping_limit_date', 'new_price',
       'freight_value', 'quantity'],
      dtype='object')

In [72]:
orders[orders['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,product_id,seller_id,shipping_limit_date,new_price,freight_value,quantity
39830,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,1.2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,78.899999,1
39831,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,1.2,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,78.899999,1
39832,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,7.8,79ce45dbc2ea29b22b5a261bbb7b7ee7,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,7.8,6.570000,1


In [73]:
customers.drop(columns='customer_unique_id', inplace = True)

In [74]:
pd.merge(orders, customers, left_on = 'customer_id', right_on = 'customer_id', how = 'inner')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,product_id,seller_id,shipping_limit_date,new_price,freight_value,quantity,customer_zip_code_prefix,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,58.900002,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.900002,13.290000,1,28013,Campos Dos Goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,239.899994,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.899994,19.930000,1,15775,Santa Fe Do Sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,199.000000,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.000000,17.870001,1,35661,Para De Minas,MG
3,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,199.899994,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.899994,18.139999,1,13226,Varzea Paulista,SP
4,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,delivered,2017-12-10 11:53:48,2017-12-10 12:10:31,2017-12-12 01:07:48,2017-12-18 22:03:38,2018-01-04,19.900000,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.900000,11.850000,1,16700,Guararapes,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78343,fffb9224b6fc7c43ebb0904318b10b5f,4d3abb73ceb86353aeadbe698aa9d5cb,delivered,2017-10-27 16:51:00,2017-10-28 02:55:58,2017-11-10 19:31:52,2017-11-17 19:41:42,2017-11-27,55.000000,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.000000,136.759995,1,56912,Serra Talhada,PE
78344,fffbee3b5462987e66fb49b1c5411df2,11a0e041ea6e7e21856d2689b64e7f3a,delivered,2018-06-19 09:27:48,2018-06-19 09:58:03,2018-06-29 13:46:00,2018-07-05 17:51:08,2018-07-23,119.849998,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.849998,20.030001,1,39401,Montes Claros,MG
78345,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,299.989990,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.989990,43.410000,1,65077,Sao Luis,MA
78346,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,350.000000,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.000000,36.529999,1,81690,Curitiba,PR


In [75]:
customers

,customer_id,customer_zip_code_prefix,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,6273,Osasco,SP
1,000161a058600d5901f007fab4c27140,35550,Itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,29830,Nova Venecia,ES
3,0002414f95344307404f0ace7a26f1d5,39664,Mendonca,MG
4,000379cdec625522490c315e70c7a9fb,4841,Sao Paulo,SP
...,...,...,...,...
99436,fffecc9f79fd8c764f843e9951b11341,95630,Parobe,RS
99437,fffeda5b6d849fbd39689bb92087f431,22461,Rio De Janeiro,RJ
99438,ffff42319e9b2d713724ae527742af25,6754,Taboao Da Serra,SP
99439,ffffa3172527f765de70084a7e53aae8,37130,Alfenas,MG


### Olist Product dataset

In [76]:
query = "select * from olist_products_dataset"
products = pd.read_sql(query, conn)

C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\11933789.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql(query, conn)


In [77]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625,20,17,13
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300,40,40,40
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700,16,19,16
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400,27,7,27
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700,31,13,20


In [78]:
products.drop(columns = ['product_name_lenght',
       'product_description_lenght', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'], inplace= True)

In [79]:
products['product_category_name'].unique()

array(['perfumaria', 'artes', 'esporte_lazer', 'bebes',
       'utilidades_domesticas', 'instrumentos_musicais', 'cool_stuff',
       'moveis_decoracao', 'eletrodomesticos', 'brinquedos',
       'cama_mesa_banho', 'construcao_ferramentas_seguranca',
       'informatica_acessorios', 'beleza_saude', 'malas_acessorios',
       'ferramentas_jardim', 'moveis_escritorio', 'automotivo',
       'eletronicos', 'fashion_calcados', 'telefonia', 'papelaria',
       'fashion_bolsas_e_acessorios', 'pcs', 'casa_construcao',
       'relogios_presentes', 'construcao_ferramentas_construcao',
       'pet_shop', 'eletroportateis', 'agro_industria_e_comercio', None,
       'moveis_sala', 'sinalizacao_e_seguranca', 'climatizacao',
       'consoles_games', 'livros_interesse_geral',
       'construcao_ferramentas_ferramentas',
       'fashion_underwear_e_moda_praia', 'fashion_roupa_masculina',
       'moveis_cozinha_area_de_servico_jantar_e_jardim',
       'industria_comercio_e_negocios', 'telefonia_fixa',
  

### product_category_name_translation

In [80]:
query = "select * from product_category_name_translation"
product_name = pd.read_sql(query, conn)

C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\2230883748.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product_name = pd.read_sql(query, conn)


In [81]:
product_name.drop(index = 0, inplace = True)

In [82]:
product_name

,Product category,Product category in english
1,beleza_saude,health_beauty
2,informatica_acessorios,computers_accessories
3,automotivo,auto
4,cama_mesa_banho,bed_bath_table
5,moveis_decoracao,furniture_decor
...,...,...
67,flores,flowers
68,artes_e_artesanato,arts_and_craftmanship
69,fraldas_higiene,diapers_and_hygiene
70,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


## Below we have 2 product category name which we done have a translation name in english

In [83]:
print(products['product_category_name'].nunique())
print(product_name['Product category'].nunique()) 

73
71


# Find the catergory which is in product table by not in Product_name table

In [84]:
product_name[~product_name['Product category'].isin(products['product_category_name'])]

,Product category,Product category in english


In [85]:
products[~products['product_category_name'].isin(product_name['Product category'])]['product_category_name'].unique()

array([None, 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
      dtype=object)

In [86]:
product_name.sort_values(by = ['Product category'])['Product category'].unique()

array(['agro_industria_e_comercio', 'alimentos', 'alimentos_bebidas',
       'artes', 'artes_e_artesanato', 'artigos_de_festas',
       'artigos_de_natal', 'audio', 'automotivo', 'bebes', 'bebidas',
       'beleza_saude', 'brinquedos', 'cama_mesa_banho', 'casa_conforto',
       'casa_conforto_2', 'casa_construcao', 'cds_dvds_musicais',
       'cine_foto', 'climatizacao', 'consoles_games',
       'construcao_ferramentas_construcao',
       'construcao_ferramentas_ferramentas',
       'construcao_ferramentas_iluminacao',
       'construcao_ferramentas_jardim',
       'construcao_ferramentas_seguranca', 'cool_stuff', 'dvds_blu_ray',
       'eletrodomesticos', 'eletrodomesticos_2', 'eletronicos',
       'eletroportateis', 'esporte_lazer', 'fashion_bolsas_e_acessorios',
       'fashion_calcados', 'fashion_esporte', 'fashion_roupa_feminina',
       'fashion_roupa_infanto_juvenil', 'fashion_roupa_masculina',
       'fashion_underwear_e_moda_praia', 'ferramentas_jardim', 'flores',
       'fral

In [87]:
product_name[product_name['Product category'] == 'pcs']

,Product category,Product category in english
54,pcs,computers


In [88]:
product_name[product_name['Product category'] == 'portateis_casa_forno_e_cafe']

,Product category,Product category in english
64,portateis_casa_forno_e_cafe,small_appliances_home_oven_and_coffee


In [89]:
products = pd.merge(products, product_name, left_on = 'product_category_name', right_on = 'Product category', how = 'left')

In [90]:
products

,product_id,product_category_name,product_photos_qty,Product category,Product category in english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,1.0,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,1.0,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,1.0,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,4.0,utilidades_domesticas,housewares
...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,2.0,moveis_decoracao,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,1.0,construcao_ferramentas_iluminacao,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,1.0,cama_mesa_banho,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,2.0,informatica_acessorios,computers_accessories


In [91]:
products[products['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos']

,product_id,product_category_name,product_photos_qty,Product category,Product category in english
5821,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN
7325,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN
8819,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos,2.0,NaN,NaN
11039,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN
14266,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN
16182,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos,4.0,NaN,NaN
17800,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos,5.0,NaN,NaN
18610,7afdd65f79f63819ff5bee328843fa37,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN
26890,a4756663d007b0cd1af865754d08d968,portateis_cozinha_e_preparadores_de_alimentos,4.0,NaN,NaN
29919,cb9d764f38ee4d0c00af64d5c388f837,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,NaN


In [92]:
 products.loc[products['product_category_name'] == 'pc_gamer', 'Product category in english'] = 'computers'
 products.loc[products['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos', 'Product category in english'] = 'small_appliances_home_oven_and_coffee'

In [93]:
products[products['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos']

,product_id,product_category_name,product_photos_qty,Product category,Product category in english
5821,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee
7325,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee
8819,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos,2.0,NaN,small_appliances_home_oven_and_coffee
11039,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee
14266,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee
16182,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos,4.0,NaN,small_appliances_home_oven_and_coffee
17800,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos,5.0,NaN,small_appliances_home_oven_and_coffee
18610,7afdd65f79f63819ff5bee328843fa37,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee
26890,a4756663d007b0cd1af865754d08d968,portateis_cozinha_e_preparadores_de_alimentos,4.0,NaN,small_appliances_home_oven_and_coffee
29919,cb9d764f38ee4d0c00af64d5c388f837,portateis_cozinha_e_preparadores_de_alimentos,1.0,NaN,small_appliances_home_oven_and_coffee


In [94]:
products['product_category_name'] = products['Product category in english']

In [95]:
products.drop(columns = ['Product category', 'Product category in english'], inplace = True)

In [96]:
products

,product_id,product_category_name,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,1.0
3,cef67bcfe19066a932b7673e239eb23d,baby,1.0
4,9dc1a7de274444849c219cff195d0b71,housewares,4.0
...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor,2.0
32947,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights,1.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table,1.0
32949,83808703fc0706a22e264b9d75f04a2e,computers_accessories,2.0


### olist_order_payments_dataset

In [97]:
query = "select * from olist_order_payments_dataset"
payments = pd.read_sql(query, conn) 

C:\Users\DELL\AppData\Local\Temp\ipykernel_6968\1334654871.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payments = pd.read_sql(query, conn)


In [98]:
payments.sort_values(by='payment_sequential', ascending = False)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52457,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.260000
52460,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.049999
18234,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.019997
45742,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.100000
92936,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.270000
...,...,...,...,...,...
103865,382a503d9d9928675c03cb26ea99b262,1,credit_card,6,61.750000
103866,d2d690bca31c33e38a006ee3b3b62c94,1,boleto,1,356.059998
103867,b5f5561bde462ba1bb5b15e5be5b6bd5,1,credit_card,2,100.239998
103868,05cd5649410dd46b0467669ff19e294b,1,credit_card,2,189.369995


In [99]:
payments[payments['order_id']=='fa65dad1b0e818e3ccc5cb0e39231352']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
2800,fa65dad1b0e818e3ccc5cb0e39231352,20,voucher,1,150.000000
8676,fa65dad1b0e818e3ccc5cb0e39231352,24,voucher,1,0.420000
10069,fa65dad1b0e818e3ccc5cb0e39231352,22,voucher,1,4.030000
18234,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.019997
23334,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.160000
27670,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.710000
30623,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.080000
32914,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.860000
36423,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.510000
38228,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.680000


In [100]:
payments.drop(columns=['payment_sequential','payment_installments'], inplace = True)

In [101]:
payments = payments.groupby(['order_id', 'payment_type'])['payment_value'].sum().reset_index()

In [102]:
print(payments['order_id'].nunique())
print(orders['order_id'].nunique())

99440
75654


In [103]:
99440-98665

775

In [104]:
payments[~payments['order_id'].isin(orders['order_id'])]['order_id'].nunique()

23787

In [105]:
orders = pd.merge(orders, payments, on='order_id', how = 'inner')

In [106]:
orders['order_id'].nunique()

75653